In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, butter, filtfilt

sns.set_theme(style="whitegrid")

RAW_SLEEP_DIR = r"E:\AVIA\data\raw\Sleep_EDF"
PROCESSED_SLEEP_DIR = r"E:\AVIA\data\processed\sleep_edf"
FIGURES_DIR = r"E:\AVIA\reports\figures"
MODELS_DIR = r"E:\AVIA\models"

os.makedirs(PROCESSED_SLEEP_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

PSG_PATH = os.path.join(RAW_SLEEP_DIR, "SC4001E0-PSG.edf")
HYP_PATH = os.path.join(RAW_SLEEP_DIR, "SC4001EC-Hypnogram.edf")

In [2]:
def read_edf_psg_and_hypnogram(psg_file, hyp_file):
    # 1. Parse PSG Header & Binary Records
    with open(psg_file, "rb") as f:
        hdr = f.read(256).decode("ascii", errors="ignore")
        n_records = int(hdr[236:244].strip())
        record_dur = float(hdr[244:252].strip())
        ns = int(hdr[252:256].strip())
        
        labels = [f.read(16).decode("ascii").strip() for _ in range(ns)]
        f.seek(256 + ns * (16 + 80 + 8))
        phys_min = np.array([float(f.read(8).decode("ascii").strip()) for _ in range(ns)])
        phys_max = np.array([float(f.read(8).decode("ascii").strip()) for _ in range(ns)])
        dig_min  = np.array([float(f.read(8).decode("ascii").strip()) for _ in range(ns)])
        dig_max  = np.array([float(f.read(8).decode("ascii").strip()) for _ in range(ns)])
        f.seek(256 + ns * (16 + 80 + 8 + 8 + 8 + 8 + 8 + 80))
        n_samps = [int(f.read(8).decode("ascii").strip()) for _ in range(ns)]
        
        header_bytes = int(hdr[184:192].strip())
        f.seek(header_bytes)
        raw_int16 = np.fromfile(f, dtype=np.int16)

    gains = (phys_max - phys_min) / (dig_max - dig_min)
    offsets = phys_min - gains * dig_min
    total_samps_per_rec = sum(n_samps)
    raw_records = raw_int16[:n_records * total_samps_per_rec].reshape(n_records, total_samps_per_rec)

    # Extract EEG Fpz-Cz (ch 0, 3000 samps/30s = 100 Hz), EEG Pz-Oz (ch 1), EOG horizontal (ch 2)
    fpz_cz = (raw_records[:, 0:3000] * gains[0] + offsets[0])        # Shape: (2650, 3000)
    pz_oz  = (raw_records[:, 3000:6000] * gains[1] + offsets[1])     # Shape: (2650, 3000)
    eog_h  = (raw_records[:, 6000:9000] * gains[2] + offsets[2])     # Shape: (2650, 3000)

    # 2. Parse EDF+ Hypnogram Annotations
    with open(hyp_file, "rb") as f:
        hyp_raw = f.read().decode("latin-1", errors="ignore")
    
    pattern = re.compile(r"\+(\d+(?:\.\d+)?)\x15(\d+(?:\.\d+)?)\x14Sleep stage ([W1234R\?])")
    matches = pattern.findall(hyp_raw)
    
    stage_per_epoch = ["?"] * n_records
    for onset_s, dur_s, stg in matches:
        start_ep = int(float(onset_s) // 30)
        n_ep     = int(float(dur_s) // 30)
        for ep in range(start_ep, min(n_records, start_ep + n_ep)):
            stage_per_epoch[ep] = stg

    return fpz_cz, pz_oz, eog_h, stage_per_epoch, labels

fpz_cz_epochs, pz_oz_epochs, eog_epochs, stage_labels, ch_names = read_edf_psg_and_hypnogram(PSG_PATH, HYP_PATH)
print(f"[+] Loaded {len(stage_labels)} 30s epochs | Channels: {ch_names[:3]}")
print("Full Recording Stage Distribution:", pd.Series(stage_labels).value_counts().to_dict())

[+] Loaded 2650 30s epochs | Channels: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal']
Full Recording Stage Distribution: {'W': 1997, '2': 250, 'R': 125, '4': 119, '3': 101, '1': 58}


In [4]:
FS_EEG = 100.0  # 100 Hz sampling rate (3,000 samples per 30-second epoch)

def compute_eeg_epoch_features(fpz_ep, pzoz_ep, eog_ep, fs=100.0):
    feats = {}
    for prefix, sig in [("fpz", fpz_ep), ("pzoz", pzoz_ep)]:
        feats[f"{prefix}_std"]      = float(np.std(sig))
        feats[f"{prefix}_kurtosis"] = float(pd.Series(sig).kurtosis())
        # Hjorth Mobility
        diff1 = np.diff(sig)
        feats[f"{prefix}_mobility"] = float(np.std(diff1) / (np.std(sig) + 1e-6))
        
        # Welch Power Spectral Density across canonical sleep frequency bands
        freqs, psd = welch(sig, fs=fs, nperseg=500)
        total_p = np.trapezoid(psd[(freqs >= 0.5) & (freqs <= 30.0)], freqs[(freqs >= 0.5) & (freqs <= 30.0)]) + 1e-8
        
        bands = {
            "delta": (0.5, 4.0),    # Deep Slow-Wave Sleep (N3)
            "theta": (4.0, 8.0),    # Light Sleep (N1/N2) & REM
            "alpha": (8.0, 12.0),   # Drowsy Wakefulness
            "sigma": (12.0, 16.0),  # Sleep Spindles (N2 hallmark)
            "beta":  (16.0, 30.0)   # Active Wake / REM
        }
        for b_name, (f_lo, f_hi) in bands.items():
            b_mask = (freqs >= f_lo) & (freqs <= f_hi)
            b_pow  = float(np.trapezoid(psd[b_mask], freqs[b_mask]))
            feats[f"{prefix}_{b_name}_rel"] = b_pow / total_p
            
        feats[f"{prefix}_delta_beta_ratio"] = feats[f"{prefix}_delta_rel"] / (feats[f"{prefix}_beta_rel"] + 1e-5)

    # EOG rapid eye movement activity (distinguishes REM from N1/Wake)
    feats["eog_std"] = float(np.std(eog_ep))
    feats["eog_max_abs"] = float(np.max(np.abs(eog_ep)))
    return feats

# Map R&K stages to AASM 5-Class Standard: 0=W, 1=N1, 2=N2, 3=N3 (Stages 3&4), 4=REM
stage_map = {"W": 0, "1": 1, "2": 2, "3": 3, "4": 3, "R": 4}
stage_names = {0: "W", 1: "N1", 2: "N2", 3: "N3_Deep", 4: "REM"}

sleep_rows = []
# Crop to In-Bed Nighttime Window (Epoch 1000 [30,000s] to Epoch 1766 [52,980s]) to avoid 16h of daytime wake
for ep_idx in range(1000, 1767):
    raw_st = stage_labels[ep_idx]
    if raw_st not in stage_map: continue
    
    row = compute_eeg_epoch_features(fpz_cz_epochs[ep_idx], pz_oz_epochs[ep_idx], eog_epochs[ep_idx], FS_EEG)
    row["subject_id"] = "SC4001E0"
    row["epoch_idx"]  = ep_idx
    row["time_sec"]   = ep_idx * 30
    row["sleep_stage_id"]   = stage_map[raw_st]
    row["sleep_stage_name"] = stage_names[stage_map[raw_st]]
    row["data_quality_ok"]  = int(row["fpz_std"] < 150.0) # Flag extreme electrode movement artifacts
    sleep_rows.append(row)

df_sleep = pd.DataFrame(sleep_rows)

# Add rolling temporal context (5-epoch rolling mean of delta/beta ratio to capture sleep stage progression)
df_sleep["fpz_delta_rolling5"] = df_sleep["fpz_delta_rel"].rolling(5, min_periods=1).mean()
df_sleep["fpz_sigma_rolling5"] = df_sleep["fpz_sigma_rel"].rolling(5, min_periods=1).mean()
df_sleep["eog_std_rolling5"]   = df_sleep["eog_std"].rolling(5, min_periods=1).mean()

sleep_csv_path = os.path.join(PROCESSED_SLEEP_DIR, "sleep_edf_sc4001_epochs_features.csv")
df_sleep.to_csv(sleep_csv_path, index=False)
print(f"[+] Saved {len(df_sleep)} In-Bed 30s Sleep Epochs -> {sleep_csv_path}")
display(df_sleep["sleep_stage_name"].value_counts())

[+] Saved 767 In-Bed 30s Sleep Epochs -> E:\AVIA\data\processed\sleep_edf\sleep_edf_sc4001_epochs_features.csv


sleep_stage_name
N2         250
N3_Deep    220
REM        125
W          114
N1          58
Name: count, dtype: int64

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, f1_score

SLEEP_FEATURE_COLS = [
    c for c in df_sleep.columns 
    if c not in ["subject_id", "epoch_idx", "time_sec", "sleep_stage_id", "sleep_stage_name", "data_quality_ok"]
]

# Chronological Block Split (First 80% of night for training, final 20% sleep cycle for validation)
split_i = int(len(df_sleep) * 0.80)
scaler_s = RobustScaler()
X_s_train = scaler_s.fit_transform(df_sleep[SLEEP_FEATURE_COLS].iloc[:split_i].values)
X_s_val   = scaler_s.transform(df_sleep[SLEEP_FEATURE_COLS].iloc[split_i:].values)
y_s_train = df_sleep["sleep_stage_id"].iloc[:split_i].values
y_s_val   = df_sleep["sleep_stage_id"].iloc[split_i:].values

class AVIASleepEncoder(nn.Module):
    def __init__(self, input_dim: int, embed_dim: int = 32, n_classes: int = 5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(64, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.SiLU()
        )
        self.stage_head = nn.Linear(embed_dim, n_classes)

    def forward(self, x):
        emb = self.encoder(x)              # 32-dim Sleep Embedding (e_sleep)
        logits = self.stage_head(emb)      # 5-class AASM sleep stage logits
        return logits, emb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
sleep_model = AVIASleepEncoder(len(SLEEP_FEATURE_COLS), 32, 5).to(device)
optimizer = torch.optim.AdamW(sleep_model.parameters(), lr=2e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

ds_s = TensorDataset(torch.tensor(X_s_train, dtype=torch.float32), torch.tensor(y_s_train, dtype=torch.long))
loader_s = DataLoader(ds_s, batch_size=32, shuffle=True)

sleep_model.train()
for epoch in range(40):
    for bx, by in loader_s:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits, _ = sleep_model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

# Export 32-Dim Embeddings & Nightly Recovery Indicators
sleep_model.eval()
X_s_all = scaler_s.transform(df_sleep[SLEEP_FEATURE_COLS].values)
with torch.no_grad():
    logits_all, emb_s_all = sleep_model(torch.tensor(X_s_all, dtype=torch.float32, device=device))
    preds_all = torch.argmax(logits_all, dim=1).cpu().numpy()
    confs_all = torch.max(torch.softmax(logits_all, dim=1), dim=1)[0].cpu().numpy()
    emb_s_np  = emb_s_all.cpu().numpy()

print(f"[+] Sleep Stage Classification Accuracy: {accuracy_score(df_sleep['sleep_stage_id'], preds_all)*100:.2f}% | Macro F1: {f1_score(df_sleep['sleep_stage_id'], preds_all, average='macro'):.4f}")

torch.save(sleep_model.state_dict(), os.path.join(MODELS_DIR, "avia_sleep_encoder.pt"))
emb_s_df = pd.DataFrame(emb_s_np, columns=[f"sleep_emb_{i}" for i in range(32)])
emb_s_df.insert(0, "subject_id", df_sleep["subject_id"])
emb_s_df.insert(1, "epoch_idx", df_sleep["epoch_idx"])
emb_s_df.insert(2, "true_stage_id", df_sleep["sleep_stage_id"])
emb_s_df.insert(3, "pred_stage_id", preds_all)
emb_s_df.insert(4, "stage_confidence", np.round(confs_all, 4))
emb_s_df.insert(5, "data_quality_ok", df_sleep["data_quality_ok"])

sleep_emb_path = os.path.join(PROCESSED_SLEEP_DIR, "sleep_edf_32dim_embeddings.csv")
emb_s_df.to_csv(sleep_emb_path, index=False)
print(f"[+] Saved 32-Dim Sleep Embeddings -> {sleep_emb_path} (Shape: {emb_s_df.shape})")

[+] Sleep Stage Classification Accuracy: 88.40% | Macro F1: 0.8401
[+] Saved 32-Dim Sleep Embeddings -> E:\AVIA\data\processed\sleep_edf\sleep_edf_32dim_embeddings.csv (Shape: (767, 38))
